# sentinel-remediate — report figures

Every number and figure here is rebuilt from the DuckDB run store by `sentinel.eval.report` / `sentinel.eval.figures` — the same code `make eval` and the console's Results screen use, so the three can never disagree.

Run `make eval` first (it retrains the learned verifier and regenerates every run from scratch). This notebook only reads.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from IPython.display import Image, Markdown, display

from sentinel.config import load_config, resolve_path
from sentinel.eval.figures import render_all
from sentinel.eval.report import build_results, to_markdown
from sentinel.store.duckdb_store import DuckDBStore

config = load_config(ROOT / "config" / "offline_eval.yaml")
ev = config["eval"]
models_dir = resolve_path(config["verifier"]["learned_model_path"]).parent
store = DuckDBStore(resolve_path(config["store"]["path"]))
results = build_results(store, ev["experiment"], resamples=ev["bootstrap_resamples"],
                        seed=config["orchestrator"]["seed"], models_dir=models_dir)
print(results["n_runs"], "runs")

In [ ]:
display(Markdown(to_markdown(results)))

In [ ]:
for path in render_all(results, ROOT / "runs" / "eval" / "figures", models_dir=models_dir):
    display(Markdown(f"**{path.stem}**"), Image(filename=str(path), width=720))

## Reading these results honestly

- **Agent.** Offline runs use the seeded `SimulatedDiagnosisAgent`, a controlled proposal generator — the results measure the *verifier* against a known proposal mix, not an LLM agent's diagnosis skill. LLM baselines run with `make eval WITH_LLM=1` (costs money).
- **Labels.** Most labels are rule-derived (`provenance` field). A type-only baseline already separates them well (see `learned_verifier_training.type_only_baseline`); the learned verifier's margin is clearest on the within-type subset.
- **Block handling.** The blueprint escalates on every block, which lowers resolution; the *Block handling* ablation (retry with the block reasons) shows that cost belongs to the escalation policy, not the verifier.